# Advanced Retrieval Pipeline with Metadata Filtering and Query Refinement

This notebook implements a retrieval system that:
1.  Analyzes user queries to extract metadata filters (Year, Dept, etc.).
2.  Refines the query for better semantic matching.
3.  Performs a filtered similarity search in ChromaDB.

In [1]:
import os
import json
from typing import List, Optional
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from pydantic import BaseModel, Field

# Load environment variables
load_dotenv("../.env")

# Configuration
DB_DIR = "../metadata_n_db/chroma_db"

if not os.path.exists(DB_DIR):
    print(f"Warning: DB directory {DB_DIR} does not exist. Please check the path.")
else:
    print(f"DB Directory: {os.path.abspath(DB_DIR)}")

DB Directory: /home/rishabh/coding/pro/RAG/metadata_n_db/chroma_db


In [2]:
class RAGRetriever:
    def __init__(self, db_dir: str, api_key_env: str = "GEMINI1", model_name: str = "gemini-1.5-flash"):
        """
        Initialize the RAG Retriever with Embeddings, Vector Store, BM25, and LLM.
        """
        self.api_key = os.getenv(api_key_env)
        if not self.api_key:
            raise ValueError(f"API Key environment variable '{api_key_env}' not found.")
            
        print(f"Initializing RAGRetriever with model: {model_name}")
        
        # 1. Initialize LLM
        self.llm = ChatGoogleGenerativeAI(
            model=model_name,
            temperature=0,
            google_api_key=self.api_key
        )
        
        # 2. Initialize Embeddings
        self.embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            google_api_key=self.api_key
        )
        
        # 3. Load Vector Store
        self.vectorstore = Chroma(
            persist_directory=db_dir,
            embedding_function=self.embeddings
        )
        print(f"Vector Store loaded with {self.vectorstore._collection.count()} documents.")
        
        # 4. Initialize BM25 (Keyword Search)
        print("Building BM25 Index from Vector Store documents...")
        self._build_bm25_index()
        
        # 5. Setup Chains
        self._setup_reranking_chain()
        self._setup_multi_query_chain()
        
    def _build_bm25_index(self):
        """
        Fetches all documents from ChromaDB to build an in-memory BM25 index.
        """
        try:
            data = self.vectorstore.get() 
            docs = []
            if data and data['documents']:
                for i, text in enumerate(data['documents']):
                    metadata = data['metadatas'][i] if data['metadatas'] else {}
                    docs.append(Document(page_content=text, metadata=metadata))
            
            if not docs:
                print("Warning: No documents found in Vector Store to build BM25 index.")
                self.bm25_retriever = None
                return

            self.bm25_retriever = BM25Retriever.from_documents(docs)
            print(f"BM25 Index built with {len(docs)} documents.")
            
        except Exception as e:
            print(f"Error building BM25 index: {e}")
            self.bm25_retriever = None

    def _setup_reranking_chain(self):
        """Sets up the LLM chain used for reranking documents."""
        class RelevanceScore(BaseModel):
            index: int = Field(description="The index of the document in the provided list")
            relevance_score: float = Field(description="A score from 0.0 to 1.0 indicating relevance")
            reasoning: str = Field(description="Brief reason why this document matches the constraints")

        class RankedDocuments(BaseModel):
            ranked_results: List[RelevanceScore]

        self.rerank_parser = JsonOutputParser(pydantic_object=RankedDocuments)

        self.rerank_prompt = PromptTemplate(
            template="""You are an expert relevance ranker. 
            The user asked: "{query}"
            
            Below is a list of document snippets retrieved from a database. 
            Your job is to evaluate each snippet and determine if it truly answers the user's specific constraints (e.g., specific year, specific department, specific format).
            
            If a document is relevant, assign a high score (0.7 - 1.0).
            If it is topic-adjacent but misses the specific constraint (e.g., wrong year), assign a low score (0.0 - 0.3).
            
            Documents:
            {doc_list}
            
            Return the output as valid JSON matching the format instructions.
            {format_instructions}
            """,
            input_variables=["query", "doc_list"],
            partial_variables={"format_instructions": self.rerank_parser.get_format_instructions()},
        )

        self.rerank_chain = self.rerank_prompt | self.llm | self.rerank_parser

    def _setup_multi_query_chain(self):
        """Sets up the LLM chain for generating multiple query variations."""
        class MultiQuery(BaseModel):
            queries: List[str] = Field(description="List of 3 alternative versions of the user query")

        self.mq_parser = JsonOutputParser(pydantic_object=MultiQuery)

        valid_departments = [
            "ARTIFICIAL INTELLIGENCE AND DATA ENGINEERING",
            "ARCHITECTURE AND PLANNING",
            "CHEMICAL ENGINEERING",
            "CIVIL ENGINEERING",
            "COMPUTER SCIENCE AND ENGINEERING",
            "ELECTRICAL ENGINEERING",
            "ELECTRONICS AND COMMUNICATION ENGINEERING",
            "MECHANICAL ENGINEERING",
            "METALLURGICAL AND MATERIALS ENGINEERING",
            "CHEMISTRY",
            "MATHEMATICS",
            "PHYSICS",
            "MANAGEMENT STUDIES"
        ]

        valid_centres = [
            "CENTRE FOR ENERGY AND ENVIRONMENT",
            "NATIONAL CENTRE FOR DISASTER MITIGATION AND MANAGEMENT",
            "MATERIALS RESEARCH CENTRE"
        ]

        self.mq_prompt = PromptTemplate(
            template="""You are an AI assistant optimizing queries for an academic document retrieval system at MNIT (Malaviya National Institute of Technology).
            
            Your task is to generate 3 different versions of the user's question to improve retrieval.
            
            CRITICAL TERMINOLOGY RULES:
            1. Replace generic terms with formal academic terminology used in Indian Institutes, following are some examples:
               - 'Teachers' / 'Professors' -> 'Faculty'
               - 'College' / 'University' / 'NIT' -> 'Institute' or 'MNIT'
               - 'Classes' -> 'Lectures' 
               - 'Exams' -> 'Examinations'
               - 'Hostel' -> 'Hostels'
            
            2. STRICTLY standardize Department names. If the user mentions a department (even abbreviations like 'CS', 'AI', 'Mech'), replace it with the EXACT full name from this list:
            {valid_departments}
            
            3. If the user mentions a Centre, use the EXACT full name from this list:
            {valid_centres}

            4. If no specific department is mentioned, use 'Institute' context.

            Original Question: {question}
            
            Return the output as valid JSON matching the format instructions.
            {format_instructions}
            """,
            input_variables=["question"],
            partial_variables={
                "format_instructions": self.mq_parser.get_format_instructions(),
                "valid_departments": json.dumps(valid_departments),
                "valid_centres": json.dumps(valid_centres)
            },
        )

        self.mq_chain = self.mq_prompt | self.llm | self.mq_parser

    def generate_queries(self, original_query: str) -> List[str]:
        """Generates 3 variations of the query using the LLM."""
        try:
            result = self.mq_chain.invoke({"question": original_query})
            queries = result.get("queries", [])
            # Ensure original query is included
            if original_query not in queries:
                queries.insert(0, original_query)
            return queries[:4] # Return max 4 queries (original + 3 generated)
        except Exception as e:
            print(f"Multi-query generation failed: {e}")
            return [original_query]

    def reciprocal_rank_fusion(self, results: List[List[Document]], k=60):
        """
        Combines multiple lists of ranked documents using Reciprocal Rank Fusion (RRF).
        """
        fused_scores = {}
        doc_map = {} 

        for rank_list in results:
            for rank, doc in enumerate(rank_list):
                doc_key = doc.page_content
                if doc_key not in fused_scores:
                    fused_scores[doc_key] = 0
                    doc_map[doc_key] = doc
                fused_scores[doc_key] += 1 / (rank + k)

        reranked_results = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
        return [doc_map[doc_str] for doc_str, score in reranked_results]

    def retrieve(self, query: str, k_fetch: int = 10, top_n: int = 3) -> List:
        """
        Retrieves documents using Multi-Query + Hybrid Search + RRF + LLM Reranking.
        """
        print(f"--- 1. Multi-Query Generation for: '{query}' ---")
        queries = self.generate_queries(query)
        print(f"Generated Queries: {queries}")
        
        all_results_lists = []
        
        print(f"\n--- 2. Hybrid Retrieval for each query ---")
        for q in queries:
            # Vector Search
            vector_docs = self.vectorstore.similarity_search(q, k=k_fetch)
            all_results_lists.append(vector_docs)
            
            # Keyword Search
            if self.bm25_retriever:
                self.bm25_retriever.k = k_fetch
                keyword_docs = self.bm25_retriever.invoke(q)
                all_results_lists.append(keyword_docs)
        
        # Fusion
        print(f"\n--- 3. RRF Fusion of {len(all_results_lists)} result lists ---")
        initial_docs = self.reciprocal_rank_fusion(all_results_lists, k=60)
        
        # Slice to keep context window reasonable
        initial_docs = initial_docs[:k_fetch * 2] # Allow slightly more docs for reranker
        
        print(f"[Log] Top {len(initial_docs)} Documents after Fusion:")
        for i, doc in enumerate(initial_docs[:5]): # Print top 5 only to avoid clutter
            print(f"  [{i}] Source: {doc.metadata.get('source')} | Title: {doc.metadata.get('title')}")
        
        # Format for LLM
        doc_texts = []
        for i, doc in enumerate(initial_docs):
            snippet = f"Doc ID {i}:\nMetadata: {doc.metadata}\nContent: {doc.page_content[:400]}..." 
            doc_texts.append(snippet)
        
        combined_text = "\n\n".join(doc_texts)

        print(f"\n--- 4. Reranking {len(initial_docs)} documents ---")
        try:
            ranking_result = self.rerank_chain.invoke({"query": query, "doc_list": combined_text})
            sorted_ranks = sorted(ranking_result['ranked_results'], key=lambda x: x['relevance_score'], reverse=True)
            
            final_docs = []
            print("\n--- Top Selected Documents ---")
            for item in sorted_ranks[:top_n]:
                if item['relevance_score'] < 0.5:
                    print(f"Skipping Doc {item['index']} (Low Score: {item['relevance_score']})")
                    continue
                    
                if 0 <= item['index'] < len(initial_docs):
                    original_doc = initial_docs[item['index']]
                    print(f"Score: {item['relevance_score']} | Doc Source: {original_doc.metadata.get('source')}")
                    print(f"Reasoning: {item['reasoning']}")
                    final_docs.append(original_doc)
                
            return final_docs

        except Exception as e:
            print(f"Reranking failed: {e}. Falling back to raw search results.")
            return initial_docs[:top_n]

In [3]:
# Initialize the Retriever
# You can switch models here (e.g., "gemini-2.5-flash-lite" if available)
retriever = RAGRetriever(
    db_dir=DB_DIR, 
    api_key_env="GEMINI1", 
    model_name="gemini-2.5-flash-lite"
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Initializing RAGRetriever with model: gemini-2.5-flash-lite
Vector Store loaded with 2339 documents.
Building BM25 Index from Vector Store documents...
Vector Store loaded with 2339 documents.
Building BM25 Index from Vector Store documents...
BM25 Index built with 2339 documents.
BM25 Index built with 2339 documents.


In [4]:
# Test 2: Syllabus
retriever.retrieve("Syllabus of Mathematics-I for first year")

--- 1. Multi-Query Generation for: 'Syllabus of Mathematics-I for first year' ---
Generated Queries: ['Syllabus of Mathematics-I for first year', 'Curriculum for MATHEMATICS for first year students at MNIT', 'Syllabus for introductory MATHEMATICS lectures at MNIT', 'Academic outline for MATHEMATICS I for undergraduate students at the Institute']

--- 2. Hybrid Retrieval for each query ---
Generated Queries: ['Syllabus of Mathematics-I for first year', 'Curriculum for MATHEMATICS for first year students at MNIT', 'Syllabus for introductory MATHEMATICS lectures at MNIT', 'Academic outline for MATHEMATICS I for undergraduate students at the Institute']

--- 2. Hybrid Retrieval for each query ---

--- 3. RRF Fusion of 8 result lists ---
[Log] Top 20 Documents after Fusion:
  [0] Source: ../pdfs/1st_Year_Scheme_Syallbus.pdf | Title: Curricular Structure for B.Tech. I Year
  [1] Source: ../pdfs/1st_Year_Scheme_Syallbus.pdf | Title: Curricular Structure for B.Tech. I Year
  [2] Source: ../pdf

[Document(id='6629a86c-ce04-4b99-86a9-8fc10173fb40', metadata={'year': 'Unknown', 'source': '../pdfs/1st_Year_Scheme_Syallbus.pdf', 'page': 4, 'moddate': '2016-04-19T09:50:49+05:30', 'creationdate': '2016-04-19T09:50:49+05:30', 'total_pages': 15, 'doc_type': 'Syllabus', 'creator': 'Microsoft® Word 2013', 'summary': 'This document outlines the curricular structure for the first year of the B.Tech. program, common to all branches at MNIT Jaipur.', 'page_label': '5', 'producer': 'Microsoft® Word 2013', 'audience': 'UG', 'author': 'Valued Customer', 'title': 'Curricular Structure for B.Tech. I Year', 'Dept': 'Institute'}, page_content='Differential Calculus :  Curvature , Concavity, convexity and points of  Inflexion, \nAsymptotes, Partial differentiation, Euler’s theorem on homogeneous functions, Total \ndifferentiation, Approximate calculation, Curve tracing (Cartesian and five polar curves - \nFolium of Descartes, Limacon, Cardioids, Lemniscates of Bernoulli and Equiangular \nspiral). \

In [5]:
# Test 3: Fee Structure (Specific Year)
retriever.retrieve("Fee Structure year 2016 admitted students for Btech students")

--- 1. Multi-Query Generation for: 'Fee Structure year 2016 admitted students for Btech students' ---
Generated Queries: ['Fee Structure year 2016 admitted students for Btech students', 'Fee structure for B.Tech students admitted in the academic year 2016 at MNIT', 'Tuition and other fees for Bachelor of Technology cohort admitted in 2016 at the Institute', 'Details of fee schedule for 2016 admitted B.Tech candidates at MNIT']

--- 2. Hybrid Retrieval for each query ---
Generated Queries: ['Fee Structure year 2016 admitted students for Btech students', 'Fee structure for B.Tech students admitted in the academic year 2016 at MNIT', 'Tuition and other fees for Bachelor of Technology cohort admitted in 2016 at the Institute', 'Details of fee schedule for 2016 admitted B.Tech candidates at MNIT']

--- 2. Hybrid Retrieval for each query ---

--- 3. RRF Fusion of 8 result lists ---
[Log] Top 20 Documents after Fusion:
  [0] Source: ../pdfs/Final_Fee_Structure_PG_2017-18_admitted.pdf | Title:

[Document(id='5cec31ff-209e-4db2-8492-8cd5b1a2ff75', metadata={'moddate': '2017-04-27T18:21:24+05:30', 'creator': 'Microsoft® Word 2013', 'doc_type': 'Fee Structure', 'title': 'Fee Structure for B. Tech. /B.Arch.', 'creationdate': '2017-04-27T18:21:24+05:30', 'author': 'IBM', 'total_pages': 3, 'audience': 'UG', 'page_label': '1', 'producer': 'Microsoft® Word 2013', 'summary': 'This document details the fee structure for B.Tech and B.Arch students admitted in the 2016-17 session.', 'page': 0, 'year': '2016', 'source': '../pdfs/Fee_Structure_UG_2016-17.pdf', 'Dept': 'Institute'}, page_content='MALAVIYA NATIONAL INSTITUTE OF TECHNOLOGY JAIPUR \n \n \n Fee Structure for B. Tech. /B.Arch. students admitted in the session 2016-17 \n \nTUITION FEE \n \nS. No. \nHead of Fee \n \nOdd Semester & Even Semester \nOP/OBC PH/SC/ST \nIncome  \nBelow 1 Lac \nIncome                                  \n1 Lac to 5 Lac \nIncome                   \nAbove 5 Lac All \n1. Tuition Fee per Semester 0 20,834.00 6

In [6]:
# Test 4: Faculty Query
retriever.retrieve("Quantum computing classes for faculty")

--- 1. Multi-Query Generation for: 'Quantum computing classes for faculty' ---
Generated Queries: ['Quantum computing classes for faculty', 'Lectures on quantum computing for faculty at MNIT', 'Quantum computing instruction for MNIT faculty', 'Academic resources for faculty regarding quantum computing at the Institute']

--- 2. Hybrid Retrieval for each query ---
Generated Queries: ['Quantum computing classes for faculty', 'Lectures on quantum computing for faculty at MNIT', 'Quantum computing instruction for MNIT faculty', 'Academic resources for faculty regarding quantum computing at the Institute']

--- 2. Hybrid Retrieval for each query ---

--- 3. RRF Fusion of 8 result lists ---
[Log] Top 20 Documents after Fusion:
  [0] Source: ../pdfs/QT-03_BASIC_QUANTUM_PROGRAMMING_2025.pdf | Title: Online Faculty Programme on Basic Quantum Programming
  [1] Source: ../pdfs/QT-05_Quantum_Computation_Brochure_updated_1.pdf | Title: Online Faculty Programme on Quantum Computation
  [2] Source: .

[Document(id='3db9341f-1307-414b-8cd8-283407fa7918', metadata={'total_pages': 1, 'summary': 'An intensive 20-day training programme on Basic Quantum Programming is being organized for faculty and students.', 'page': 0, 'producer': 'www.ilovepdf.com', 'moddate': '2025-05-15T05:14:13+00:00', 'Dept': 'Institute', 'audience': 'Faculty', 'title': 'Online Faculty Programme on Basic Quantum Programming', 'year': '2025', 'creationdate': '2025-05-15T05:14:12+00:00', 'page_label': '1', 'doc_type': 'Notice', 'author': 'x', 'creator': 'Microsoft® Word 2016', 'source': '../pdfs/QT-03_BASIC_QUANTUM_PROGRAMMING_2025.pdf'}, page_content='AICTE Approved Minor Course Curriculum \non Quantum Computing \n                  \n  \n \nhttp://www.mnit.ac.in/eict \nOnline Faculty Programme on  \nBasic Quantum \nProgramming  \nMay 16- June 6, 2025 \nTwenty Days (Mon to Sat) \nTime: 2 – 4 PM (Daily 2 Hours) \n \n \nAn intensive 20 Day - 40 Hours Training Programme in Online Mode is being \norganized for faculty a

In [8]:
# Test 5: Policy
retriever.retrieve("What is the unfair means policy policy or UFM of the instituion")

--- 1. Multi-Query Generation for: 'What is the unfair means policy policy or UFM of the instituion' ---
Generated Queries: ['What is the unfair means policy policy or UFM of the instituion', 'What is the Unfair Means (UFM) policy of MNIT?', "Kindly provide information on the Institute's policy regarding Unfair Means (UFM) in examinations.", 'Details regarding the Unfair Means (UFM) regulations at MNIT are requested.']

--- 2. Hybrid Retrieval for each query ---
Generated Queries: ['What is the unfair means policy policy or UFM of the instituion', 'What is the Unfair Means (UFM) policy of MNIT?', "Kindly provide information on the Institute's policy regarding Unfair Means (UFM) in examinations.", 'Details regarding the Unfair Means (UFM) regulations at MNIT are requested.']

--- 2. Hybrid Retrieval for each query ---

--- 3. RRF Fusion of 8 result lists ---
[Log] Top 20 Documents after Fusion:
  [0] Source: ../pdfs/sou_motu_n.pdf | Title: Organisation and Function
  [1] Source: ../pdfs

[Document(metadata={'moddate': '2023-09-18T19:17:31+05:30', 'year': '2023', 'creationdate': '2023-09-18T19:17:31+05:30', 'Dept': 'Institute', 'summary': 'The National Testing Agency is releasing admit cards for the Stage-I Recruitment Examination for Non-Teaching positions at NITs and other Institutions under the Ministry of Education.', 'author': 'admin', 'creator': 'Microsoft® Word 2021', 'audience': 'UG', 'page_label': '1', 'title': 'Release of Admit Card s to the Candidates for Stage-I Recruitment Examination for Non-Teaching positions at NITs and Other Institutions under Ministry of Education', 'page': 0, 'producer': 'Microsoft® Word 2021', 'doc_type': 'Public Notice', 'total_pages': 1, 'source': '../pdfs/Public-Notice-Release-of-Admit-Cards-CRENIT-2023.pdf'}, page_content='• Candidate must not mutilate the Admit Card or change any entry made therein. Any \ntampering in the particulars, photograph, signature, thumb impression in this Admit \nCard shall be considered as Unfair Mean

In [9]:
# Test 5: Policy
retriever.retrieve("what are guidlines for phd exam in year 2025")

--- 1. Multi-Query Generation for: 'what are guidlines for phd exam in year 2025' ---
Generated Queries: ['what are guidlines for phd exam in year 2025', 'Guidelines for Doctoral Examinations at MNIT for the year 2025', 'PhD Examination Regulations MNIT 2025', 'MNIT Ph.D. Examination Policies and Procedures 2025']

--- 2. Hybrid Retrieval for each query ---
Generated Queries: ['what are guidlines for phd exam in year 2025', 'Guidelines for Doctoral Examinations at MNIT for the year 2025', 'PhD Examination Regulations MNIT 2025', 'MNIT Ph.D. Examination Policies and Procedures 2025']

--- 2. Hybrid Retrieval for each query ---

--- 3. RRF Fusion of 8 result lists ---
[Log] Top 20 Documents after Fusion:
  [0] Source: ../pdfs/Guidelines.pdf | Title: Ph.D. Entrance Exam Guidelines
  [1] Source: ../pdfs/SynopsisSubmissionRequirments.pdf | Title: Synopsis and Thesis Submission Guidelines
  [2] Source: ../pdfs/Guidelines.pdf | Title: Ph.D. Entrance Exam Guidelines
  [3] Source: ../pdfs/07_CE

[Document(id='367df7d2-d7d6-44b5-a1ed-9b4710d0a514', metadata={'page_label': '1', 'doc_type': 'Guidelines', 'total_pages': 1, 'creationdate': '2025-12-02T14:48:48+05:30', 'source': '../pdfs/Guidelines.pdf', 'year': '2025-26', 'moddate': '2025-12-02T14:48:48+05:30', 'title': 'Ph.D. Entrance Exam Guidelines', 'Dept': 'Institute', 'producer': 'Microsoft® Word 2013', 'page': 0, 'audience': 'UG', 'summary': 'This document outlines the guidelines for the Ph.D. entrance exam for the Even Semester 2025-26, including the selection process and requirements for shortlisted candidates.', 'creator': 'Microsoft® Word 2013', 'author': 'wipro'}, page_content='Guidelines for Ph.D. Entrance Exam, EVEN Semester 2025-26 \n \nWritten exam and interview for Ph.D. entrance exam, Even Semester 2025-26 will be \nheld during 09th and 10th December 2025 at MNIT campus.  \n \nSelection process will comprise of two steps (i) Written test (ii) Interview of \nshortlisted candidates. The written test will comprise of

In [10]:
# Test 5: Policy
retriever.retrieve("what is the college's perspective on Right to Information")

--- 1. Multi-Query Generation for: 'what is the college's perspective on Right to Information' ---
Generated Queries: ["what is the college's perspective on Right to Information", "What is MNIT's policy on the Right to Information Act?", "What is the Institute's stance on Right to Information requests?", 'What is the official perspective of Malaviya National Institute of Technology regarding the Right to Information Act?']

--- 2. Hybrid Retrieval for each query ---
Generated Queries: ["what is the college's perspective on Right to Information", "What is MNIT's policy on the Right to Information Act?", "What is the Institute's stance on Right to Information requests?", 'What is the official perspective of Malaviya National Institute of Technology regarding the Right to Information Act?']

--- 2. Hybrid Retrieval for each query ---

--- 3. RRF Fusion of 8 result lists ---
[Log] Top 20 Documents after Fusion:
  [0] Source: ../pdfs/sou_motu_n.pdf | Title: Organisation and Function
  [1] S

[Document(id='0b66e958-9e65-4747-a40e-a80111cb7b6c', metadata={'moddate': '2011-04-19T16:58:10+05:30', 'title': 'RTI Application Form', 'audience': 'General', 'creationdate': '2011-04-19T16:58:10+05:30', 'creator': 'PScript5.dll Version 5.2.2', 'page': 0, 'author': 'cse', 'page_label': '1', 'total_pages': 1, 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'summary': 'This is an application form for requesting information under the Right to Information Act, 2005 at MNIT Jaipur.', 'year': 'Unknown', 'doc_type': 'Application Form', 'source': '../pdfs/RTI-APPLICATION-FORM.pdf', 'Dept': 'Institute'}, page_content='APPLICATION FORM (RIGHT TO INFORMATION ACT, 2005)  \nTo  \nThe Central Public Information Officer(……………………………………) \nMNIT JAIPUR  \nJLN Marg Jaipur – 302017.   \n \n1. Name of the Applicant: ___________________________________________________________   \n2. Address with PIN: ________________________________________________________________ \n  ______________________________________

In [11]:
retriever.retrieve("who were the toppers in year 2021-22")


--- 1. Multi-Query Generation for: 'who were the toppers in year 2021-22' ---
Generated Queries: ['who were the toppers in year 2021-22', 'Academic performance leaders in MNIT for the academic year 2021-22', 'Identify high-achieving students at MNIT during the 2021-22 academic session', 'Retrieval of top student rankings for MNIT in 2021-22']

--- 2. Hybrid Retrieval for each query ---
Generated Queries: ['who were the toppers in year 2021-22', 'Academic performance leaders in MNIT for the academic year 2021-22', 'Identify high-achieving students at MNIT during the 2021-22 academic session', 'Retrieval of top student rankings for MNIT in 2021-22']

--- 2. Hybrid Retrieval for each query ---

--- 3. RRF Fusion of 8 result lists ---
[Log] Top 20 Documents after Fusion:
  [0] Source: ../pdfs/18thScroll_2024.pdf | Title: Scroll of Awardees
  [1] Source: ../pdfs/18thScroll_2024.pdf | Title: Scroll of Awardees
  [2] Source: ../pdfs/18thScroll_2024.pdf | Title: Scroll of Awardees
  [3] Source

[Document(metadata={'total_pages': 3, 'doc_type': 'Notice', 'moddate': '2023-03-31T16:08:35+05:30', 'summary': 'This document lists the recipients of the Director’s Gold Medal in B.Tech. and B.Arch. for the academic session 2021-22.', 'page_label': '1', 'author': 'IBM', 'creationdate': '2023-03-31T16:08:35+05:30', 'creator': 'Microsoft® Word 2013', 'title': 'Director’s Gold Medal', 'producer': 'Microsoft® Word 2013', 'page': 0, 'audience': 'UG', 'Dept': 'Institute', 'year': '2021-22', 'source': '../pdfs/Gold_Medalist.pdf'}, page_content='1 \n \n \n \n \nMALAVIYA NATIONAL INSTITUTE OF TECHNOLOGY JAIPUR \n \n \nDirector’s Gold Medal in B. Tech. and B. Arch. in the academic session \n2021-22 \n \nS. \nNo ID No. Name Programme CGPA \n1 2017UAR1567 NUPUR MALIK ARCHITECTURE AND \nPLANNING 9.41 \n2 2018UCH1656 DARSHANA \nPALIWAL CHEMICAL ENGINEERING 9.66 \n3 2018UCE1103 ARSHIKA TOMAR CIVIL ENGINEERING 9.70 \n4 2018UCP1444 PRANSHU VYAS COMPUTER SCIENCE AND \nENGINEERING 9.63 \n5 2018UEE1021 RI

In [12]:
retriever.retrieve("faculy program for smart grid")

--- 1. Multi-Query Generation for: 'faculy program for smart grid' ---
Generated Queries: ['faculy program for smart grid', 'Faculty development programs for smart grid technologies at MNIT', 'Academic initiatives for faculty in the domain of smart grids within the Institute', 'Faculty training opportunities concerning smart grid applications at Malaviya National Institute of Technology']

--- 2. Hybrid Retrieval for each query ---
Generated Queries: ['faculy program for smart grid', 'Faculty development programs for smart grid technologies at MNIT', 'Academic initiatives for faculty in the domain of smart grids within the Institute', 'Faculty training opportunities concerning smart grid applications at Malaviya National Institute of Technology']

--- 2. Hybrid Retrieval for each query ---

--- 3. RRF Fusion of 8 result lists ---
[Log] Top 20 Documents after Fusion:
  [0] Source: ../pdfs/Smarthealthcare-onlineSept2025_1.pdf | Title: Smart Healthcare Technologies
  [1] Source: ../pdfs/Q

[Document(id='aab30861-7a1a-4d0b-ba7f-7ffc5dd76d7c', metadata={'summary': 'Online Faculty Development Programme on Fundamentals of Smart Grid from July 7th to August 1st, 2025, organized by MNIT Jaipur and other institutions.', 'page': 0, 'doc_type': 'Notice', 'title': 'Fundamentals of Smart Grid', 'producer': 'Microsoft® Word for Microsoft 365', 'total_pages': 1, 'page_label': '1', 'audience': 'Faculty', 'author': 'Narasimharaju B L', 'creator': 'Microsoft® Word for Microsoft 365', 'year': '2025', 'moddate': '2025-07-23T12:43:16+05:30', 'creationdate': '2025-07-23T12:43:16+05:30', 'source': '../pdfs/_Final_Schedule_FDP_Fundamentals_of_Smart_Grid.pdf', 'Dept': 'Electronics & ICT Academy'}, page_content='MALAVIYA NATIONAL INSTITUTE OF TECHNOLOGY JAIPUR \nOnline Faculty Development Programme \n \nJointly Organized by Electronics & ICT Academy MNIT Jaipur, IIT Roorkee, IIITDM Jabalpur, NIT Patna \non \n   Fundamentals of Smart Grid \n                                          (7th July – 1